# Conditional Workflow: Data-Quality Gate

This notebook demonstrates a conditional workflow.

Scenario:

A dataset is uploaded for an ML project.

The workflow first validates the dataset. If the dataset is usable, it trains a model. If not, it generates a cleaning request.

This is a conditional workflow because the next step depends on the validation output.

## Setup

In [20]:
import os
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel
from typing import Optional, Literal, List, Dict, Any

from picoagents import Agent, OpenAIChatCompletionClient

# Load .env from the repository root or parent directory.
# Adjust this path if your notebook is located elsewhere.
load_dotenv(Path.cwd() / ".." / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    print("API key loaded successfully.")
else:
    print("OPENAI_API_KEY is empty. Deterministic workflow examples can still run, but LLM-agent examples need an API key.")

client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY
)



from picoagents.workflow import Workflow, WorkflowRunner, FunctionStep
from picoagents.workflow.core import WorkflowMetadata, StepMetadata, Context

runner = WorkflowRunner()

API key loaded successfully.


## Define typed input and output models

In [21]:
class DatasetInput(BaseModel):
    """Input schema for dataset metadata used in validation."""
    dataset_name: str
    n_rows: int
    missing_rate: float
    target_column_exists: bool


class ValidationOutput(BaseModel):
    """Validation result that drives conditional workflow routing."""
    dataset_name: str
    is_valid: bool
    reason: str
    n_rows: int
    missing_rate: float


class TrainingOutput(BaseModel):
    """Output message produced when training is allowed."""
    result: str


class CleaningRequestOutput(BaseModel):
    """Output message produced when data cleaning is required."""
    result: str

## Define workflow step functions

In [22]:
async def validate_dataset(input_data: DatasetInput, context: Context) -> ValidationOutput:
    """Validate dataset quality gates before downstream processing."""
    if input_data.n_rows < 100:
        return ValidationOutput(
            dataset_name=input_data.dataset_name,
            is_valid=False,
            reason="Dataset has fewer than 100 rows.",
            n_rows=input_data.n_rows,
            missing_rate=input_data.missing_rate
        )

    if input_data.missing_rate > 0.30:
        return ValidationOutput(
            dataset_name=input_data.dataset_name,
            is_valid=False,
            reason="Missing-value rate is above 30%.",
            n_rows=input_data.n_rows,
            missing_rate=input_data.missing_rate
        )

    if not input_data.target_column_exists:
        return ValidationOutput(
            dataset_name=input_data.dataset_name,
            is_valid=False,
            reason="Target column is missing.",
            n_rows=input_data.n_rows,
            missing_rate=input_data.missing_rate
        )

    return ValidationOutput(
        dataset_name=input_data.dataset_name,
        is_valid=True,
        reason="Dataset passed all validation checks.",
        n_rows=input_data.n_rows,
        missing_rate=input_data.missing_rate
    )


async def train_model(input_data: ValidationOutput, context: Context) -> TrainingOutput:
    """Create the training action message for valid datasets."""
    return TrainingOutput(
        result=f"Training started for {input_data.dataset_name}. Validation reason: {input_data.reason}"
    )


async def request_data_cleaning(input_data: ValidationOutput, context: Context) -> CleaningRequestOutput:
    """Create the cleaning request message for invalid datasets."""
    return CleaningRequestOutput(
        result=f"Cleaning required for {input_data.dataset_name}. Reason: {input_data.reason}"
    )

## Build conditional workflow

### Workflow Diagram: add_step vs add_edge

`add_step(...)` creates nodes in the workflow graph.
`add_edge(from, to, condition=...)` creates conditional arrows between nodes.

```mermaid
flowchart
    A[validate_dataset] -->|is_valid == True| B[train_model]
    A -->|is_valid == False| C[request_cleaning]
```

Quick code mapping:
- `add_step(...)`: register `validate`, `train`, and `cleaning` nodes.
- `add_edge(...)`: define routing rules from validation output.
- `set_start_step("validate_dataset")`: marks where execution starts.
- `add_end_step(...)`: marks branch terminal nodes.

In [23]:
validate_step = FunctionStep(
    step_id="validate_dataset",
    metadata=StepMetadata(name="Validate Dataset"),
    input_type=DatasetInput,
    output_type=ValidationOutput,
    func=validate_dataset
)

train_step = FunctionStep(
    step_id="train_model",
    metadata=StepMetadata(name="Train Model"),
    input_type=ValidationOutput,
    output_type=TrainingOutput,
    func=train_model
)

cleaning_step = FunctionStep(
    step_id="request_cleaning",
    metadata=StepMetadata(name="Request Data Cleaning"),
    input_type=ValidationOutput,
    output_type=CleaningRequestOutput,
    func=request_data_cleaning
)

conditional_workflow = (
    Workflow(metadata=WorkflowMetadata(name="Conditional Data Quality Gate"))
    # add_step: register nodes in the workflow graph
    .add_step(validate_step)
    .add_step(train_step)
    .add_step(cleaning_step)
    # add_edge: route based on validation output field values
    .add_edge("validate_dataset", "train_model", condition={
        "type": "output_based",
        "field": "is_valid",
        "operator": "==",
        "value": True
    })
    .add_edge("validate_dataset", "request_cleaning", condition={
        "type": "output_based",
        "field": "is_valid",
        "operator": "==",
        "value": False
    })
    .set_start_step("validate_dataset")
    .add_end_step("train_model")
    .add_end_step("request_cleaning")
)

## Run two examples

In [24]:
valid_dataset = {
    "dataset_name": "student_performance.csv",
    "n_rows": 1500,
    "missing_rate": 0.08,
    "target_column_exists": True
}

invalid_dataset = {
    "dataset_name": "tiny_dataset.csv",
    "n_rows": 45,
    "missing_rate": 0.04,
    "target_column_exists": True
}

print("=== Valid dataset ===")
async for event in runner.run_stream(conditional_workflow, valid_dataset):
    print(event)

print("\n=== Invalid dataset ===")
async for event in runner.run_stream(conditional_workflow, invalid_dataset):
    print(event)

=== Valid dataset ===
[21:05:48] 🚀 Workflow started with input: {'dataset_name': 'student_performance.csv', 'n_rows': 1500, 'missing_rate': 0.08, 'target_column_exists': True}
[21:05:48] ▶️  Step 'validate_dataset' started
[21:05:48] ✅ Step 'validate_dataset' completed → {'dataset_name': 'student_performance.csv', 'is_valid': True, 'reason': 'Dataset passed all validation checks.', 'n_rows': 1500, 'missing_rate': 0.08}
[21:05:48] 🔗 validate_dataset → train_model
[21:05:48] 🔗 validate_dataset → request_cleaning
[21:05:48] ▶️  Step 'train_model' started
[21:05:48] ✅ Step 'train_model' completed → {'result': 'Training started for student_performance.csv. Validation reason: Dataset passed all validation checks.'}
[21:05:48] ✅ Workflow completed in 0.00s (2 steps)

=== Invalid dataset ===
[21:05:48] 🚀 Workflow started with input: {'dataset_name': 'tiny_dataset.csv', 'n_rows': 45, 'missing_rate': 0.04, 'target_column_exists': True}
[21:05:49] ▶️  Step 'validate_dataset' started
[21:05:49] ✅ 

## Reflection questions

1. Which output field controls the routing decision?
2. How does this differ from a normal `if` statement in Python?
3. Where would this pattern be useful in an MLOps pipeline?